In [ ]:
import numpy as np
import torch
import copy
import matplotlib.pyplot as plt
from envs.multi_agent_env import StagHuntEnv

# ── 유틸리티 함수 ────────────────────────────────────────────────────────────

def patch_env_for_cloning(env):
    """환경 인스턴스에 상태 저장/복원 기능을 동적으로 추가합니다."""
    def get_env_state():
        return copy.deepcopy(env.__dict__)
    
    def set_env_state(state):
        env.__dict__.update(copy.deepcopy(state))
        
    env.get_env_state = get_env_state
    env.set_env_state = set_env_state
    return env

def obs_dict_to_tensor(obs_dict, device):
    """환경에서 반환된 obs dict를 torch tensor dict로 변환합니다."""
    return {
        i: torch.tensor(v, dtype=torch.float32, device=device)
        for i, v in obs_dict.items()
    }

def get_w_x(obs_dict, policy, agent_i, device):
    """
    W^i_t, X^i_t를 구성합니다.
    W^i_t = O_t^{-i}
    X^i_t = (O_t^i, H_{t-1}^{-i})
    """
    n = len(obs_dict)
    w_parts = [
        obs_dict[j] if isinstance(obs_dict[j], torch.Tensor)
        else torch.tensor(obs_dict[j], dtype=torch.float32, device=device)
        for j in range(n) if j != agent_i
    ]
    w_t_i = torch.cat(w_parts, dim=-1)
    obs_i = obs_dict[agent_i] if isinstance(obs_dict[agent_i], torch.Tensor) \
        else torch.tensor(obs_dict[agent_i], dtype=torch.float32, device=device)
    h_others_parts = [
        policy.agents[j].rnn_state_policy.view(-1)
        for j in range(n) if j != agent_i
    ]
    h_others = torch.cat(h_others_parts, dim=-1)
    x_t_i = torch.cat([obs_i, h_others], dim=-1)
    return w_t_i, x_t_i

# ── 환경 초기화 ──────────────────────────────────────────────────────────────
env_config = {
    "num_agents": 3,
    "max_steps": 50,
    "stag_move_prob": 0.0,
    "coop_bonus": 0.0,
    "forage_reward": 3.0,
    "stag_reward": 5.0,
    "mauling_punishment": 0.0,
    "grid_observation": True,
    "obs_noise_std": 0.1
}

env = patch_env_for_cloning(StagHuntEnv(**env_config))
device = 'cpu'
print("✅ 환경 및 유틸리티 함수 초기화 완료")


In [ ]:
# ── 헬퍼 함수: On-policy 평가 (실시간 시뮬레이션) ───────────────────────────
def load_all_models_on_policy(checkpoint_path, naive_checkpoint_path, env, device,
                             alg_num_agents=3, alg_action_dim=5,
                             alg_hidden_dim=64, alg_recurrent_N=1):
    from algorithms.r_pci import rPCIAlgorithm
    from algorithms.networks import NaivePredictor
    import os
    
    obs_dim = env.get_obs_dim()
    x_dim = obs_dim + (alg_num_agents - 1) * alg_hidden_dim * alg_recurrent_N

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    algorithm_state = checkpoint['algorithm_state']

    policy = rPCIAlgorithm(
        env=env, num_agents=alg_num_agents, obs_dim=obs_dim, action_dim=alg_action_dim,
        w_dim=obs_dim * (alg_num_agents - 1), hidden_dim=alg_hidden_dim, recurrent_N=alg_recurrent_N, device=device,
    )
    for agent_id, agent_state in algorithm_state['agents'].items():
        policy.agents[agent_id].policy_net.load_state_dict(agent_state['policy_net'])
        policy.agents[agent_id].policy_net.eval()
    policy.outcome_bridge_net.load_state_dict(algorithm_state['outcome_bridge_net'])
    policy.outcome_bridge_net.eval()

    naive_predictor = None
    if naive_checkpoint_path and os.path.exists(naive_checkpoint_path):
        naive_predictor = NaivePredictor(
            action_dim=alg_action_dim, num_agents=alg_num_agents, x_dim=x_dim, hidden_dim=alg_hidden_dim
        ).to(device)
        naive_checkpoint = torch.load(naive_checkpoint_path, map_location=device, weights_only=False)
        naive_predictor.load_state_dict(naive_checkpoint['naive_predictor_state'])
        naive_predictor.eval()
    
    return policy, policy.outcome_bridge_net, naive_predictor

def run_evaluation_on_policy(policy, outcome_bridge, naive_predictor, env, device, 
                             num_episodes=5, K_mc=30):
    """
    각 체크포인트 모델이 직접 환경에서 궤적을 생성하며 실시간으로 MC 샘플링을 수행합니다.
    (주의: 이 방식은 매우 느립니다.)
    """
    mse_pci_list = []
    mse_naive_list = []

    for ep in range(num_episodes):
        obs = env.reset()
        policy.reset_agents()
        done = False
        step = 0

        while not done and step < 50:
            obs_tensor = obs_dict_to_tensor(obs, device)
            saved_env_state = env.get_env_state()
            
            # RNN 은닉 상태 저장 (Intervention 후 복원을 위해)
            saved_rnn_states = {}
            for i in range(policy.num_agents):
                saved_rnn_states[i] = (
                    policy.agents[i].rnn_state_policy.clone(),
                    policy.agents[i].prev_action.clone()
                )

            # 모든 에이전트와 모든 행동에 대해 Intervention 평가
            for agent_i in range(policy.num_agents):
                w_t_i, x_t_i = get_w_x(obs_tensor, policy, agent_i, device)
                
                for a in range(policy.action_dim):
                    # 1. Ground Truth 계산 (MC Sampling)
                    mc_rewards = []
                    for _ in range(K_mc):
                        env.set_env_state(saved_env_state)
                        # 에이전트 RNN 상태 복원
                        for idx, (rs, pa) in saved_rnn_states.items():
                            policy.agents[idx].rnn_state_policy = rs.clone()
                            policy.agents[idx].prev_action = pa.clone()

                        actions_dict = policy.get_actions(obs_tensor)
                        actions_env = {j: actions_dict[j][0] for j in range(policy.num_agents)}
                        actions_env[agent_i] = a # 강제 행동 주입
                        
                        _, rewards, _, _ = env.step(actions_env)
                        mc_rewards.append(np.sum(list(rewards.values())))
                    
                    mu_true = np.mean(mc_rewards)

                    # 2. 모델 예측값과 비교
                    a_tensor = torch.tensor([a], dtype=torch.long, device=device)
                    w_tensor = w_t_i.unsqueeze(0)
                    x_tensor = x_t_i.unsqueeze(0)
                    
                    with torch.no_grad():
                        r_hat_all = outcome_bridge.forward(a_tensor, w_tensor, x=x_tensor)
                        pci_est = r_hat_all[0].sum().item()
                        mse_pci_list.append((mu_true - pci_est)**2)
                        
                        if naive_predictor is not None:
                            r_hat_naive = naive_predictor.forward(a_tensor, x_tensor)
                            naive_est = r_hat_naive[0].sum().item()
                            mse_naive_list.append((mu_true - naive_est)**2)

            # 실제 환경 진행 (현재 모델의 정책에 따라 한 스텝 이동)
            env.set_env_state(saved_env_state)
            for idx, (rs, pa) in saved_rnn_states.items():
                policy.agents[idx].rnn_state_policy = rs.clone()
                policy.agents[idx].prev_action = pa.clone()

            actions_dict = policy.get_actions(obs_tensor)
            actions_env = {j: actions_dict[j][0] for j in range(policy.num_agents)}
            obs, _, dones, _ = env.step(actions_env)
            done = all(dones) if isinstance(dones, (list, np.ndarray)) else dones
            step += 1
            
        print(f"  - 에피소드 {ep+1}/{num_episodes} 완료")
            
    return {
        'pci_mse': np.mean(mse_pci_list),
        'naive_mse': np.mean(mse_naive_list) if mse_naive_list else None
    }


In [ ]:
# ── 온폴리씨 평가 실행 ──────────────────────────────────────────────────────
pci_base_dir = "save_checkpoints/20260316_152607_seed43/staghuntenv/checkpoints"
naive_base_dir = "save_checkpoints/20260316_152607_seed43/staghuntenv/checkpoints_naivepredictor"

target_itrs = [0, 500, 1000, 1499]
on_policy_results = {}

for itr in target_itrs:
    pci_filename = "final_checkpoint.pt" if itr == 1499 else f"checkpoint_{itr}.pt"
    naive_filename = "final_checkpoint.pt" if itr == 1499 else f"checkpoint_{itr}.pt"
    
    pci_path = f"{pci_base_dir}/{pci_filename}"
    naive_path = f"{naive_base_dir}/{naive_filename}"
    
    print(f"🔍 Iteration {itr} 온폴리씨 평가 중... (실시간 시뮬레이션)")
    pol, ob, nv = load_all_models_on_policy(pci_path, naive_path, env, device)
    
    # 실시간으로 MC 샘플링을 수행하므로 시간이 오래 걸립니다. (num_episodes를 작게 설정)
    metrics = run_evaluation_on_policy(pol, ob, nv, env, device, num_episodes=5, K_mc=30)
    on_policy_results[itr] = metrics
    
    msg = f"  → PCI MSE: {metrics['pci_mse']:.4f}"
    if metrics['naive_mse'] is not None:
        msg += f" | Naive MSE: {metrics['naive_mse']:.4f}"
    print(msg)

print("\n✅ 모든 체크포인트 온폴리씨 평가 완료!")


In [ ]:
# ── 결과 시각화 ──────────────────────────────────────────────────────────────
itrs = sorted(on_policy_results.keys())
pci_mses = [on_policy_results[i]['pci_mse'] for i in itrs]
naive_mses = [on_policy_results[i]['naive_mse'] for i in itrs]

plt.figure(figsize=(10, 5))
plt.plot(itrs, pci_mses, marker='o', color='blue', label='PCI Estimator (On-policy)', linewidth=2)
if all(m is not None for m in naive_mses):
    plt.plot(itrs, naive_mses, marker='s', color='gray', linestyle='--', label='Naive Predictor (On-policy)', linewidth=2)

plt.xlabel("Training Iterations")
plt.ylabel("Mean Squared Error (MSE)")
plt.title("Error Reduction (On-policy Evaluation: Individual Trajectories)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
